# TP 4 — Le neurone et l'apprentissage

**M1 · Deep Learning & Big Data · Séance 4 · Environnement : TensorFlow Playground puis Google Colab (GPU)**

| Partie | Contenu | Durée indicative |
|---|---|---|
| A | Comprendre un réseau en jouant avec TensorFlow Playground | 25 min |
| B | Classer des critiques de films avec un réseau Keras (données du TP 3) | 35 min |
| C | Observer et combattre le surapprentissage | 25 min |

Complétez les `____` des cellules `# TODO` et répondez aux questions ❓.

---
## Partie A — TensorFlow Playground

Ouvrez <https://playground.tensorflow.org> dans un nouvel onglet. Rien à installer : un vrai réseau de neurones s'entraîne dans votre navigateur.

**Repères :**
- À gauche, les **données** : des points orange et bleus à séparer.
- Au centre, le **réseau** : les entrées (`X1`, `X2`), les couches cachées (**+** / **−** pour en ajouter), les neurones de chaque couche.
- Les traits entre neurones sont les **poids** : épaisseur = importance, couleur = signe.
- À droite, la **sortie** et la *Test loss* : l'erreur sur des points que le réseau n'a pas appris.
- Le bouton ▶ lance l'entraînement ; le compteur *Epoch* défile.

### 🛠️ Expérience A.1 — Un seul neurone

Choisissez le jeu de données **en deux paquets** (*Gaussian*, en bas à gauche des quatre jeux), **zéro couche cachée**, puis ▶.
Le réseau sépare-t-il les deux couleurs ? Quelle forme a la frontière ?

### 🛠️ Expérience A.2 — Le cercle

Choisissez le jeu **cercle** (*Circle*, en haut à gauche), toujours sans couche cachée. Puis ajoutez **une couche cachée de 4 neurones**.
Que se passe-t-il dans chaque cas ?

### 🛠️ Expérience A.3 — La spirale

Choisissez la **spirale** (*Spiral*, en bas à droite). Trouvez une architecture qui atteint une *Test loss* inférieure à 0,1. Notez votre meilleure configuration.

### 🛠️ Expérience A.4 — Le taux d'apprentissage

Sur le jeu cercle avec une couche de 4 neurones, essayez un *Learning rate* de **0,0001**, puis **0,03**, puis **3**. Décrivez ce que vous observez.

### 🛠️ Expérience A.5 — Le surapprentissage

Jeu **deux paquets**, *Ratio of training to test data* à **10 %**, *Noise* à **50**, 3 couches de 8 neurones. Lancez longtemps. Comparez *Training loss* et *Test loss*.

### ❓ Notez vos observations

| Expérience | Ce que j'observe | Ce que j'en retiens |
|---|---|---|
| A.1 | | |
| A.2 | | |
| A.3 | | |
| A.4 | | |
| A.5 | | |

---
## Partie B — Classer des critiques de films

On reprend les critiques IMDb nettoyées au TP 3. Objectif : prédire si une critique est **positive (1)** ou **négative (0)**.

**Avant tout :** *Exécution › Modifier le type d'exécution › GPU*.

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
import matplotlib.pyplot as plt

print("GPU :", tf.config.list_physical_devices("GPU") or "aucun")

In [ ]:
# Chargement du jeu préparé au TP 3 (dossier local ou Google Drive), sinon solution de repli
candidats = ["imdb_propre", "/content/drive/MyDrive/DeepLearning/imdb_propre"]

if not any(os.path.isdir(c) for c in candidats):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

chemin = next((c for c in candidats if os.path.isdir(c)), None)

if chemin:
    train_df = pd.read_parquet(os.path.join(chemin, "split=train"))
    test_df = pd.read_parquet(os.path.join(chemin, "split=test"))
    print("Données du TP 3 chargées depuis", chemin)
else:
    # Repli : on recharge IMDb et on applique un nettoyage minimal
    !pip install -q datasets
    from datasets import load_dataset
    imdb = load_dataset("stanfordnlp/imdb")
    def nettoyer(d):
        d = d.rename(columns={"text": "texte_propre"})
        d["texte_propre"] = (d["texte_propre"].str.replace(r"<[^>]+>", " ", regex=True)
                             .str.lower().str.replace(r"\s+", " ", regex=True).str.strip())
        return d
    train_df = nettoyer(imdb["train"].to_pandas())
    test_df = nettoyer(imdb["test"].to_pandas())
    print("Repli : IMDb rechargé depuis Hugging Face")

print(len(train_df), "critiques d'entraînement ·", len(test_df), "critiques de test")
train_df[["texte_propre", "label"]].head()

In [ ]:
# On garde 10 % de l'entraînement pour la validation
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
n_val = len(train_df) // 10
val_df, train_df = train_df[:n_val], train_df[n_val:]

def vers_dataset(d, melanger=False):
    ds = tf.data.Dataset.from_tensor_slices((d["texte_propre"].values, d["label"].values.astype("float32")))
    if melanger:
        ds = ds.shuffle(10_000, seed=42)
    return ds.batch(256)

train_ds, val_ds, test_ds = vers_dataset(train_df, True), vers_dataset(val_df), vers_dataset(test_df)
print("Entraînement :", len(train_df), "· validation :", len(val_df), "· test :", len(test_df))

### Du texte aux nombres : le sac de mots

Un réseau ne lit que des nombres. Méthode la plus simple :
1. on construit un **vocabulaire** des mots les plus fréquents ;
2. chaque critique devient un vecteur de 0 et de 1 : **1 si le mot du vocabulaire apparaît dans la critique**.

On perd l'ordre des mots (« pas bon du tout » et « tout du bon pas » donnent le même vecteur), mais c'est un excellent point de départ.

### 🛠️ Exercice B.1 — Le vectoriseur

Créez un vocabulaire des **10 000 mots** les plus fréquents.

In [ ]:
TAILLE_VOCAB = ____   # TODO : nombre de mots du vocabulaire

vectoriseur = layers.TextVectorization(max_tokens=TAILLE_VOCAB, output_mode="multi_hot")
vectoriseur.adapt(train_df["texte_propre"].values)   # apprend le vocabulaire

vocab = vectoriseur.get_vocabulary()
print("Taille du vocabulaire :", len(vocab))
print("20 mots les plus fréquents :", vocab[:20])

In [ ]:
# À quoi ressemble une critique vectorisée ?
exemple = vectoriseur(tf.constant(["this movie was great and the actors were great"]))
print("Forme :", exemple.shape, "· nombre de 1 :", int(exemple.numpy().sum()))

# On applique le vectoriseur à la volée, lot par lot, pour ne pas saturer la mémoire
def vectoriser(x, y):
    return vectoriseur(x), y

train_v = train_ds.map(vectoriser).prefetch(tf.data.AUTOTUNE)
val_v = val_ds.map(vectoriser).prefetch(tf.data.AUTOTUNE)
test_v = test_ds.map(vectoriser).prefetch(tf.data.AUTOTUNE)

### ❓ Question B.2

Pourquoi le vecteur ne contient-il que 6 ou 7 fois la valeur 1 alors que la phrase compte 9 mots ?

*Votre réponse :*

### 🛠️ Exercice B.3 — Le modèle

- Entrée : un vecteur de `TAILLE_VOCAB` valeurs ;
- une couche cachée de **16 neurones** (`relu`) ;
- une sortie à **1 neurone** avec l'activation `sigmoid`, qui donne une probabilité entre 0 et 1 (« probabilité que la critique soit positive »).

Pour une sortie oui / non, la perte s'appelle `binary_crossentropy`.

In [ ]:
modele = keras.Sequential([
    keras.Input(shape=(TAILLE_VOCAB,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(____, activation=____),        # TODO : nombre de neurones et activation de sortie
])

modele.compile(optimizer="adam", loss=____, metrics=["accuracy"])   # TODO : la perte
modele.summary()

In [ ]:
historique = modele.fit(train_v, validation_data=val_v, epochs=5)

In [ ]:
perte, precision = modele.evaluate(test_v, verbose=0)
print(f"Bonnes réponses sur le test : {precision:.1%}")

### 🛠️ Exercice B.4 — Testez vos propres critiques

Écrivez quelques critiques **en anglais** (le modèle n'a appris que l'anglais), dont une ambiguë ou ironique.

In [ ]:
mes_critiques = [
    "an absolute masterpiece, i loved every minute",
    "boring, too long and the acting was terrible",
    ____,   # TODO : votre critique
]
probas = modele.predict(vectoriseur(tf.constant(mes_critiques)), verbose=0).ravel()
for texte, p in zip(mes_critiques, probas):
    print(f"{p:6.1%} positif  ←  {texte}")

### ❓ Question B.5

Trouvez une phrase sur laquelle le modèle se trompe. Pourquoi le sac de mots a-t-il du mal avec elle ?

*Votre réponse :*

---
## Partie C — Observer et combattre le surapprentissage

### 🛠️ Exercice C.1 — Provoquer le surapprentissage

On prend un modèle **plus gros** (deux couches de 64 neurones) et on l'entraîne **20 époques**.

In [ ]:
gros = keras.Sequential([
    keras.Input(shape=(TAILLE_VOCAB,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
gros.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
hist_gros = gros.fit(train_v, validation_data=val_v, epochs=____)   # TODO : 20 époques

In [ ]:
def tracer(hist, titre):
    plt.figure(figsize=(7, 4))
    plt.plot(hist.history["loss"], marker="o", label="Erreur entraînement")
    plt.plot(hist.history["val_loss"], marker="o", label="Erreur validation")
    plt.xlabel("Époque"); plt.ylabel("Perte"); plt.title(titre)
    plt.legend(); plt.grid(alpha=0.3); plt.show()

tracer(hist_gros, "Gros modèle, 20 époques")

### ❓ Question C.2

À partir de quelle époque l'erreur de validation remonte-t-elle ? Que fait le modèle pendant ce temps ?

*Votre réponse :*

### 🛠️ Exercice C.3 — Deux remèdes

1. **Dropout** : pendant l'entraînement, on « éteint » au hasard une partie des neurones à chaque lot. Le réseau ne peut plus compter sur un neurone en particulier, il doit généraliser. Mettez un taux de **0,5**.
2. **Arrêt anticipé** (*early stopping*) : on arrête l'entraînement quand l'erreur de validation ne s'améliore plus depuis 2 époques, et on garde les meilleurs poids. Surveillez `val_loss`.

In [ ]:
regularise = keras.Sequential([
    keras.Input(shape=(TAILLE_VOCAB,)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(____),                    # TODO : taux de dropout
    layers.Dense(64, activation="relu"),
    layers.Dropout(____),                    # TODO
    layers.Dense(1, activation="sigmoid"),
])
regularise.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

arret = keras.callbacks.EarlyStopping(monitor=____, patience=2, restore_best_weights=True)   # TODO
hist_reg = regularise.fit(train_v, validation_data=val_v, epochs=20, callbacks=[arret])
tracer(hist_reg, "Avec dropout et arrêt anticipé")

In [ ]:
for nom, m in [("Petit modèle", modele), ("Gros modèle 20 époques", gros), ("Régularisé", regularise)]:
    print(f"{nom:<25} {m.evaluate(test_v, verbose=0)[1]:.1%}")

### 🛠️ Exercice C.4 — Regarder les erreurs

Affichez 5 critiques de test mal classées par le modèle régularisé.

In [ ]:
probas_test = regularise.predict(test_v, verbose=0).ravel()
test_df = test_df.assign(proba=probas_test, prediction=(probas_test > 0.5).astype(int))
erreurs = test_df[test_df["prediction"] != test_df["label"]]
print(f"{len(erreurs)} erreurs sur {len(test_df)}")
for _, ligne in erreurs.head(5).iterrows():
    print(f"\nVrai : {ligne['label']} · prédit : {ligne['proba']:.0%} positif\n{ligne['texte_propre'][:300]}…")

---
## 🎯 Pour vérifier que vous avez compris

1. Qu'apporte une couche cachée par rapport à un neurone seul ?
2. Que se passe-t-il si le taux d'apprentissage est trop grand ?
3. Citez deux moyens de lutter contre le surapprentissage.

*Vos réponses :*